# AR and MIDAS Baselines vs the LSTM

Paper-adapted baselines (Ball & Ghysels 2018, *Management Science* 64(10):4936–4952) for the annual size-normalized targets, extended with a MIDAS variant that consumes the full LSTM feature set.

## 1. Setup & Data

### 1.1 Configuration

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from IPython.display import display

warnings.filterwarnings('ignore', category=RuntimeWarning)
SEED = 42
DATA_PATH = None
for _cand in ['Merged_Dataset_yoy.csv',
              'Dataset/Processed/Merged_Dataset_yoy.csv',
              '../Dataset/Processed/Merged_Dataset_yoy.csv',
              '../../Dataset/Processed/Merged_Dataset_yoy.csv']:
    if Path(_cand).exists():
        DATA_PATH = Path(_cand)
        break
if DATA_PATH is None:
    raise FileNotFoundError('Merged_Dataset_yoy.csv not found (tried cwd, repo root, Model/).')

TRAIN_END = 2022
VALIDATION_YEAR = 2023
HOLDOUT_YEARS = [2024, 2025]
ROLLING_START_YEAR = 2012
MIN_TRAIN_ROWS = 40
MAX_AR_LAG = 3
MAX_MIDAS_YEAR_LAG = 0
MIN_DMSFE_HISTORY = 4
DMSFE_LOOKBACK = 8
DMSFE_DELTA = 0.95
MAX_COMBO_WEIGHT = 0.5
CLIP_PREDICTIONS = True
RUN_RIDGE = True
DENOM_REL_FLOOR = 0.10  # denominator credibility floor

TARGETS = [
    ('ebitda_ns', 'EBITDA margin', 'EBITDA', 'Net_Sales'),
    ('ni_ns', 'net margin', 'Net_Income', 'Net_Sales'),
    ('ni_ta', 'ROA', 'Net_Income', 'Total_Assets'),
]

MIDAS_PREDICTORS = [
    'excess_return', 'volatility', 'IT_Inflation', 'Brent_Close',
    'Euribor_3M', 'term_spread', 'NFCI', 'YC__curv',
]
print('Using:', DATA_PATH.resolve())

### 1.2 Annual panel: credibility-filtered, winsorized Δ-ratio targets

In [ ]:
full = pd.read_csv(DATA_PATH, low_memory=False)
full['Date'] = pd.to_datetime(full['Date'])
full = full.sort_values(['Company', 'Date']).reset_index(drop=True)
full['year'] = full['Date'].dt.year.astype(int)
full['month'] = full['Date'].dt.month.astype(int)

duplicate_daily = int(full.duplicated(['Company', 'Date']).sum())
if duplicate_daily:
    print(f'Warning: dropping {duplicate_daily} duplicate company-date rows; keeping the last row.')
full = full.drop_duplicates(['Company', 'Date'], keep='last').reset_index(drop=True)

annual_raw = full[full['EBITDA'].notna()].copy()
annual_raw = annual_raw.sort_values(['Company', 'Date'])
duplicate_annual = int(annual_raw.duplicated(['Company', 'year']).sum())
if duplicate_annual:
    print(f'Warning: dropping {duplicate_annual} duplicate company-year target rows; keeping the last row.')
annual = annual_raw.drop_duplicates(['Company', 'year'], keep='last').copy()


def credibility_mask(frame, den_col):
    _den = frame[den_col].astype(float)
    _tmp = frame[["Company"]].assign(_den=_den)
    pos_med = _tmp[_tmp["_den"] > 0].groupby("Company")["_den"].median()
    med = frame["Company"].map(pos_med)
    return (_den > 0) & (_den >= DENOM_REL_FLOOR * med)


def winsor_limits(frame, numerator, denominator, mask=None):
    denominator_ok = credibility_mask(frame, denominator)
    if mask is not None:
        denominator_ok = denominator_ok & np.asarray(mask, dtype=bool)
    raw = np.where(denominator_ok, frame[numerator].astype(float) / frame[denominator].astype(float), np.nan)
    lo, hi = np.nanpercentile(raw, [0.5, 99.5])
    return float(lo), float(hi)


def add_ratio(frame, key, numerator, denominator, lo=None, hi=None):
    denominator_ok = credibility_mask(frame, denominator)
    raw = np.where(denominator_ok, frame[numerator].astype(float) / frame[denominator].astype(float), np.nan)
    if lo is None:
        lo, hi = np.nanpercentile(raw, [0.5, 99.5])
    frame[key] = np.clip(raw, lo, hi)
    frame[f'{key}_prev'] = frame.groupby('Company')[key].shift(1)
    frame[f'd{key}'] = frame[key] - frame[f'{key}_prev']
    frame[f'd{key}_prev'] = frame.groupby('Company')[f'd{key}'].shift(1)


for key, label, numerator, denominator in TARGETS:
    add_ratio(annual, key, numerator, denominator)
    _cm = credibility_mask(annual, denominator)
    _nc = int(((annual[denominator].astype(float) > 0) & ~_cm).sum())
    print(f'{label}: non-credible-denominator rows set NaN = {_nc}')
annual_global = annual


def build_fold_frame(frame, train_end, winsor_mode='global'):
    if winsor_mode == 'global':
        return frame
    train_mask = frame['year'] <= train_end
    out = frame.copy()
    for key, _, numerator, denominator in TARGETS:
        lo, hi = winsor_limits(frame, numerator, denominator, mask=train_mask)
        add_ratio(out, key, numerator, denominator, lo=lo, hi=hi)
    return out


print(f'Annual panel: {len(annual)} rows, {annual.Company.nunique()} companies, years {annual.year.min()}-{annual.year.max()}')
for key, label, _, _ in TARGETS:
    cols = [f'd{key}', f'd{key}_prev', f'{key}_prev']
    print(f'{label}: usable observations = {int(annual[cols].notna().all(axis=1).sum())}')

EBITDA margin: non-credible-denominator rows set NaN = 32
net margin: non-credible-denominator rows set NaN = 32
ROA: non-credible-denominator rows set NaN = 12
Annual panel: 2411 rows, 178 companies, years 2009-2025
EBITDA margin: usable observations = 2005
net margin: usable observations = 2005
ROA: usable observations = 2034


### 1.3 Paper-8 monthly predictor panel (Ball–Ghysels analogs)

In [3]:
daily = full.copy()
daily['logret'] = daily.groupby('Company')['Close'].transform(lambda s: np.log(s / s.shift(1)))
daily['market_logret'] = np.log(daily['FTMIB_Close'] / daily.groupby('Company')['FTMIB_Close'].shift(1))
daily['year_month'] = daily['Date'].dt.to_period('M')

company_month = (daily.groupby(['Company', 'year_month'], as_index=False)
    .agg(company_logret=('logret', 'sum'),
         market_logret=('market_logret', 'sum'),
         volatility=('logret', lambda s: np.nanmean(np.square(s.to_numpy(float)))),
         IT_Inflation=('IT_Inflation', 'last'),
         Brent_Close=('Brent_Close', 'last'),
         Euribor_3M=('Euribor_3M', 'last'),
         IT10YT_Yield=('IT10YT_Yield', 'last'),
         NFCI=('NFCI', 'last'),
         YC__curv=('YC__curv', 'last')))
company_month['excess_return'] = company_month['company_logret'] - company_month['market_logret']
company_month['term_spread'] = company_month['IT10YT_Yield'] - company_month['Euribor_3M']
company_month['year'] = company_month['year_month'].dt.year.astype(int)
company_month['month'] = company_month['year_month'].dt.month.astype(int)

monthly = company_month[['Company', 'year', 'month'] + MIDAS_PREDICTORS].copy()
monthly = monthly.replace([np.inf, -np.inf], np.nan)
print('Monthly panel:', len(monthly), 'rows')
print(monthly.groupby('year').size().tail().to_string())

Monthly panel: 29565 rows
year
2021    2046
2022    2138
2023    2203
2024    2226
2025    2232


## 2. Baselines

- AR: BIC-selected lags (1–3) of the Δ-ratio itself.
- MIDAS / MIDAS_RIDGE: one model per monthly predictor (AR lags + 12 monthly lags, U-MIDAS); ridge guards against the ill-conditioning of slow-moving levels (see §7).
- MIDAS_RIDGE_DMSFE / _EQUAL: DMSFE-weighted (δ=0.95, 8-year lookback, 0.5 cap) and equal-weight combinations of the 8 per-predictor forecasts.
- Persistence (no-change): Δ = 0, i.e. ratio_t = ratio_{t−1}; deterministic reference.

### 2.1 Model selection & design builders

In [4]:
annual_lookup = annual_global.set_index(['Company', 'year'])
monthly_lookup = monthly.set_index(['Company', 'year', 'month'])


def build_monthly_wide(predictor):
    s = monthly_lookup[predictor].reset_index()
    w = s.pivot_table(index=['Company', 'year'], columns='month', values=predictor)
    w.columns = [f'm{mm}' for mm in range(1, 13)]
    return w.reset_index()


MONTHLY_WIDE = {p: build_monthly_wide(p) for p in MIDAS_PREDICTORS}


def build_design(rows, target_key, predictor, ar_lag, midas_year_lag, annual_lookup, monthly_lookup):
    y_col = f'd{target_key}'
    df = rows[['Company', 'year', y_col]].dropna(subset=[y_col]).copy()
    if df.empty:
        return np.empty((0, 0)), np.empty(0), []
    df = df.sort_values(['Company', 'year']).reset_index(drop=True)
    al = annual_lookup[[y_col]].reset_index().rename(columns={y_col: 'value'})
    for lag in range(1, ar_lag + 1):
        lag_key = al.assign(year=al['year'] + lag)
        df[f'lag{lag}'] = df[['Company', 'year']].merge(lag_key, on=['Company', 'year'], how='left')['value']
    if predictor is not None:
        w = MONTHLY_WIDE[predictor]
        for year_lag in range(midas_year_lag + 1):
            wk = w.assign(year=w['year'] + year_lag)
            m = df[['Company', 'year']].merge(wk, on=['Company', 'year'], how='left')
            for mm in range(1, 13):
                df[f'y{year_lag}_m{mm}'] = m[f'm{mm}']
    cols = [c for c in df.columns if c.startswith('lag') or re.search(r'_m\d+$', c)]
    X = df[cols].to_numpy(float)
    valid = np.isfinite(X).all(axis=1) & np.isfinite(df[y_col].to_numpy(float))
    keys_out = [(c, int(y)) for c, y in zip(df['Company'].to_numpy()[valid], df['year'].to_numpy()[valid])]
    return X[valid], df[y_col].to_numpy(float)[valid], keys_out


def bic_score(X, y):
    if len(y) <= X.shape[1] + 2:
        return np.inf
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    residuals = y - X @ beta
    rss = max(float(residuals @ residuals), 1e-12)
    return len(y) * np.log(rss / len(y)) + X.shape[1] * np.log(len(y))


def _fit_common(model, clip_lo, clip_hi):
    return {**model, 'clip_lo': float(clip_lo), 'clip_hi': float(clip_hi)}


def fit_bic_ols(train_rows, target_key, predictor, annual_lookup, monthly_lookup):
    candidates = []
    for ar_lag in range(1, MAX_AR_LAG + 1):
        year_lags = range(MAX_MIDAS_YEAR_LAG + 1) if predictor is not None else [0]
        for midas_year_lag in year_lags:
            X, y, _ = build_design(train_rows, target_key, predictor, ar_lag, midas_year_lag, annual_lookup, monthly_lookup)
            if len(y) == 0:
                continue
            X = np.column_stack([np.ones(len(X)), X])
            score = bic_score(X, y)
            candidates.append((score, ar_lag, midas_year_lag, X, y))
    if not candidates:
        return None
    _, ar_lag, midas_year_lag, X, y = min(candidates, key=lambda item: item[0])
    if len(y) < MIN_TRAIN_ROWS or np.linalg.matrix_rank(X) < X.shape[1]:
        return None
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    clip_lo, clip_hi = np.nanpercentile(y, [1, 99])
    return _fit_common({'intercept': float(beta[0]), 'coef': beta[1:], 'ar_lag': ar_lag,
                        'midas_year_lag': midas_year_lag, 'n_train': len(y), 'bic': bic_score(X, y),
                        'method': 'ols'}, clip_lo, clip_hi)


def fit_ridge_cv(train_rows, target_key, predictor, annual_lookup, monthly_lookup):
    best = None
    for ar_lag in range(1, MAX_AR_LAG + 1):
        year_lags = range(MAX_MIDAS_YEAR_LAG + 1) if predictor is not None else [0]
        for midas_year_lag in year_lags:
            X, y, _ = build_design(train_rows, target_key, predictor, ar_lag, midas_year_lag, annual_lookup, monthly_lookup)
            if len(y) <= X.shape[1] + 2:
                continue
            cv = RidgeCV(alphas=np.logspace(-3, 3, 15)).fit(X, y)
            if best is None or -cv.best_score_ < best[0]:
                best = (-cv.best_score_, ar_lag, midas_year_lag, X, y, cv)
    if best is None:
        return None
    _, ar_lag, midas_year_lag, X, y, cv = best
    if len(y) < MIN_TRAIN_ROWS:
        return None
    clip_lo, clip_hi = np.nanpercentile(y, [1, 99])
    return _fit_common({'intercept': float(cv.intercept_), 'coef': cv.coef_, 'ar_lag': ar_lag,
                        'midas_year_lag': midas_year_lag, 'n_train': len(y), 'bic': best[0],
                        'alpha': float(cv.alpha_), 'method': 'ridge'}, clip_lo, clip_hi)


def predict_rows(model, rows, target_key, predictor, annual_lookup, monthly_lookup):
    if model is None:
        return np.full(len(rows), np.nan)
    X, _, keys = build_design(rows, target_key, predictor, model['ar_lag'], model['midas_year_lag'], annual_lookup, monthly_lookup)
    if len(X) == 0:
        return np.full(len(rows), np.nan)
    predictions = dict(zip(keys, model['intercept'] + X @ model['coef']))
    out = np.asarray([predictions.get((r['Company'], int(r['year'])), np.nan) for _, r in rows.iterrows()])
    if CLIP_PREDICTIONS:
        out = np.clip(out, model['clip_lo'], model['clip_hi'])
    return out

### 2.2 Expanding walk-forward forecasts (rolling 2012–2025, train-only winsorization)

In [5]:
def make_forecasts(target_key, predictor=None, years=None, train_end_override=None,
                   winsor_mode='global', ridge=False):
    y_col = f'd{target_key}'
    fit_fn = fit_ridge_cv if (ridge and predictor is not None) else fit_bic_ols
    model_label = 'AR' if predictor is None else ('MIDAS_RIDGE' if ridge else 'MIDAS')
    output, selections = [], []
    for year in years:
        train_end = year - 1 if train_end_override is None else train_end_override
        frame = build_fold_frame(annual_global, train_end, winsor_mode)
        fold_annual_lookup = frame.set_index(['Company', 'year'])
        target_rows = frame[['Company', 'year', y_col]].dropna().copy()
        train_rows = target_rows[target_rows['year'] <= train_end]
        test_rows = target_rows[target_rows['year'] == year]
        if len(train_rows) < MIN_TRAIN_ROWS or test_rows.empty:
            continue
        model = fit_fn(train_rows, target_key, predictor, fold_annual_lookup, monthly_lookup)
        if model is None:
            continue
        preds = predict_rows(model, test_rows, target_key, predictor, fold_annual_lookup, monthly_lookup)
        for (_, row), pred in zip(test_rows.iterrows(), preds):
            output.append({'target': target_key, 'model': model_label,
                           'predictor': predictor or 'lagged_target', 'Company': row['Company'],
                           'year': int(year), 'actual': float(row[y_col]), 'prediction': float(pred),
                           'winsor': winsor_mode, 'method': model['method']})
        selections.append({'target': target_key, 'model': model_label,
                           'predictor': predictor or 'lagged_target', 'year': int(year), **model})
    return pd.DataFrame(output), pd.DataFrame(selections)


def run_model_set(years=None, train_end_override=None, winsor_mode='global', ridge=False):
    forecasts, selections = [], []
    for target_key, _, _, _ in TARGETS:
        if not ridge:
            f, s = make_forecasts(target_key, None, years, train_end_override, winsor_mode, ridge=False)
            forecasts.append(f); selections.append(s)
        for predictor in MIDAS_PREDICTORS:
            f, s = make_forecasts(target_key, predictor, years, train_end_override, winsor_mode, ridge=ridge)
            forecasts.append(f); selections.append(s)
    return pd.concat(forecasts, ignore_index=True), pd.concat(selections, ignore_index=True)


rolling_forecasts, rolling_selections = run_model_set(
    years=range(ROLLING_START_YEAR, int(annual_global.year.max()) + 1), winsor_mode='train'
)
if RUN_RIDGE:
    rolling_forecasts_r, rolling_selections_r = run_model_set(
        years=range(ROLLING_START_YEAR, int(annual_global.year.max()) + 1), winsor_mode='train', ridge=True
    )
    rolling_forecasts = pd.concat([rolling_forecasts, rolling_forecasts_r], ignore_index=True)
    rolling_selections = pd.concat([rolling_selections, rolling_selections_r], ignore_index=True)
print('Rolling forecasts:', len(rolling_forecasts))
display(rolling_selections.groupby(['model', 'predictor'])['ar_lag'].agg(['count', 'mean']).round(2))

Rolling forecasts: 94685


count  mean
model       predictor                 
AR          lagged_target     42  1.00
MIDAS       Brent_Close       39  1.00
            Euribor_3M        36  1.00
            IT_Inflation      12  1.00
            NFCI              36  1.00
            YC__curv          39  1.00
            excess_return     42  1.00
            term_spread       39  1.00
            volatility        42  1.00
MIDAS_RIDGE Brent_Close       42  1.98
            Euribor_3M        42  2.12
            IT_Inflation      42  2.12
            NFCI              42  2.12
            YC__curv          42  2.10
            excess_return     42  2.19
            term_spread       42  2.12
            volatility        42  2.26

### 2.3 Forecast combinations (DMSFE / equal) + persistence

In [ ]:
def add_combinations(forecasts, midas_model='MIDAS_RIDGE', history=None):
    forecasts = forecasts.copy()
    history = forecasts if history is None else history
    combined = []
    for target_key, _, _, _ in TARGETS:
        midas = forecasts[forecasts['target'].eq(target_key) & forecasts['model'].eq(midas_model)]
        hist_midas = history[history['target'].eq(target_key) & history['model'].eq(midas_model)]
        for year in sorted(midas['year'].unique()):
            history_years = [y for y in sorted(hist_midas['year'].unique()) if y < year][-DMSFE_LOOKBACK:]
            scores = {}
            for predictor in MIDAS_PREDICTORS:
                hist = hist_midas[hist_midas['predictor'].eq(predictor) & hist_midas['year'].isin(history_years)]
                if hist.empty:
                    continue
                by_year = (hist.assign(sq_error=(hist['actual'] - hist['prediction']) ** 2)
                           .dropna(subset=['sq_error']).groupby('year')['sq_error'].mean())
                score = sum(DMSFE_DELTA ** (len(history_years) - i) * by_year.get(y, np.nan)
                            for i, y in enumerate(history_years) if np.isfinite(by_year.get(y, np.nan)))
                if np.isfinite(score):
                    scores[predictor] = score
            pool = [p for p, v in scores.items() if v > 0]
            if not pool:
                continue
            if len(history_years) >= MIN_DMSFE_HISTORY:
                inverse = {p: 1.0 / scores[p] for p in pool}
                total = sum(inverse.values())
                weights = {p: w / total for p, w in inverse.items()}
                if max(weights.values()) > MAX_COMBO_WEIGHT:
                    weights = {p: min(w, MAX_COMBO_WEIGHT) for p, w in weights.items()}
                    wsum = sum(weights.values())
                    weights = {p: w / wsum for p, w in weights.items()}
            else:
                weights = {p: 1.0 for p in pool}
            current = midas[midas['year'].eq(year)]
            for company, group in current.groupby('Company'):
                group = group.set_index('predictor')
                valid = [p for p in weights if p in group.index and np.isfinite(group.loc[p, 'prediction'])]
                if not valid:
                    continue
                actual = float(group.loc[valid[0], 'actual'])
                wsum = sum(weights[p] for p in valid)
                dmsfe_pred = sum(weights[p] * group.loc[p, 'prediction'] for p in valid) / wsum
                equal_pred = float(group.loc[valid, 'prediction'].mean())
                combined.append({'target': target_key, 'model': f'{midas_model}_DMSFE', 'predictor': 'combination',
                                 'Company': company, 'year': int(year), 'actual': actual,
                                 'prediction': dmsfe_pred, 'n_predictors': len(valid)})
                combined.append({'target': target_key, 'model': f'{midas_model}_EQUAL', 'predictor': 'combination',
                                 'Company': company, 'year': int(year), 'actual': actual,
                                 'prediction': equal_pred, 'n_predictors': len(valid)})
    return pd.concat([forecasts, pd.DataFrame(combined)], ignore_index=True)


def add_persistence(forecasts):
    # Persistence (no-change) reference: predicts d_ratio = 0 for every
    # (Company, year) with an actual, i.e. ratio_t = ratio_{t-1}. Deterministic,
    # no fitted parameters — the same no-change baseline used by the LSTM notebooks.
    base = forecasts.dropna(subset=['actual'])[['target', 'Company', 'year', 'actual']].drop_duplicates()
    if base.empty:
        return forecasts
    persist = base.assign(model='Persistence (no-change)', predictor='no_change',
                          prediction=0.0, winsor='n/a', method='none')
    for col in forecasts.columns:
        if col not in persist.columns:
            persist[col] = np.nan
    return pd.concat([forecasts, persist[forecasts.columns]], ignore_index=True)


rolling_forecasts = add_combinations(rolling_forecasts)
rolling_forecasts = add_persistence(rolling_forecasts)
print(rolling_forecasts['model'].value_counts().to_string())

model
MIDAS_RIDGE                47456
MIDAS                      41297
AR                          5932
Persistence (no-change)     5932
MIDAS_RIDGE_DMSFE           5121
MIDAS_RIDGE_EQUAL           5121


### 2.4 Evaluation machinery

In [7]:
HEADLINE_MODELS = ['AR', 'MIDAS_RIDGE_DMSFE', 'MIDAS_RIDGE_EQUAL', 'Persistence (no-change)']


def common_keys(f, target):
    tf = f[f['target'].eq(target)].dropna(subset=['actual', 'prediction'])
    per = {m: set(zip(g['Company'], g['year'])) for m, g in tf.groupby('model')}
    return set.intersection(*per.values()) if per else set()

def metric_table_predictor(forecasts, years):
    f = forecasts[forecasts['year'].isin(years) & forecasts['model'].isin(['MIDAS', 'MIDAS_RIDGE'])].copy()
    f = f.dropna(subset=['actual', 'prediction'])
    ar = (forecasts[forecasts['model'].eq('AR') & forecasts['year'].isin(years)]
          [['Company', 'year', 'prediction']].rename(columns={'prediction': 'ar_prediction'})
          .dropna(subset=['ar_prediction']).drop_duplicates(['Company', 'year']))
    rows = []
    for (target, model, predictor), group in f.groupby(['target', 'model', 'predictor']):
        merged = group.merge(ar, on=['Company', 'year'], how='inner')
        if merged.empty:
            continue
        error = merged['prediction'].to_numpy() - merged['actual'].to_numpy()
        numerator = np.median(np.abs(merged['actual'] - merged['prediction']))
        denominator = np.median(np.abs(merged['actual'] - merged['ar_prediction']))
        rows.append({'target': target, 'model': model, 'predictor': predictor, 'n': len(merged),
                     'MAE': np.mean(np.abs(error)),
                     'corr': np.corrcoef(merged['actual'], merged['prediction'])[0, 1] if len(merged) > 1 else np.nan,
                     'MABER_vs_AR': numerator / denominator if denominator > 0 else np.nan})
    return pd.DataFrame(rows)


## 3. Richer MIDAS: full LSTM feature set (30 predictors)

The paper-8 panel uses only the 8 high-frequency analogs of Ball & Ghysels (2018). This section reruns the same methodology — one U-MIDAS ridge model per predictor (AR lags + 12 monthly lags), then DMSFE/EQUAL combination on the full information set used by RNN models.

### 3.1 Rich monthly panel

In [ ]:
RICH_LEVEL = [
    'DE10YT_Yield', 'IT10YT_Yield', 'EURUSD_Close', 'EURUSD_Volume',
    'FTMIB_Close', 'FTMIB_Volume', 'GVZ_Close', 'Brent_Close', 'Brent_Volume',
    'OVX_Close', 'VIX_Close', 'Gold_Close', 'YC__level', 'YC__slope',
    'YC__curv', 'TTF_Gas_Close', 'NFCI',
    'Euribor_3M', 'IT_Inflation', 'IT_Unemployment', 'PCOPPUSDM', 'PALUMUSDM',
    'IT_GDP',
]
RICH_SUM = ['Close_logret', 'ff_HML', 'ff_Mom']
RICH_MEAN = ['Volume_log']

if 'Close_logret' not in daily.columns:
    daily['Close_logret'] = daily.groupby('Company')['Close'].transform(lambda s: np.log(s / s.shift(1)))
if 'Volume_log' not in daily.columns:
    daily['Volume_log'] = np.log1p(daily['Volume'].clip(lower=0))

_rich_named = {'company_logret': ('logret', 'sum'),
               'market_logret': ('market_logret', 'sum'),
               'volatility': ('logret', lambda s: np.nanmean(np.square(s.to_numpy(float))))}
_rich_named.update({c: (c, 'last') for c in RICH_LEVEL})
_rich_named.update({c: (c, 'sum') for c in RICH_SUM})
_rich_named.update({c: (c, 'mean') for c in RICH_MEAN})
company_month_rich = daily.groupby(['Company', 'year_month'], as_index=False).agg(**_rich_named)
company_month_rich['excess_return'] = company_month_rich['company_logret'] - company_month_rich['market_logret']
company_month_rich['term_spread'] = company_month_rich['IT10YT_Yield'] - company_month_rich['Euribor_3M']
company_month_rich['year'] = company_month_rich['year_month'].dt.year.astype(int)
company_month_rich['month'] = company_month_rich['year_month'].dt.month.astype(int)

RICH_MIDAS_PREDICTORS = RICH_LEVEL + RICH_SUM + RICH_MEAN + ['excess_return', 'volatility', 'term_spread']
assert set(MIDAS_PREDICTORS).issubset(RICH_MIDAS_PREDICTORS), 'rich set must contain the paper-8'
print(f'Rich predictors ({len(RICH_MIDAS_PREDICTORS)}): {RICH_MIDAS_PREDICTORS}')

rich_monthly = company_month_rich[['Company', 'year', 'month'] + RICH_MIDAS_PREDICTORS].replace([np.inf, -np.inf], np.nan)
rich_monthly_lookup = rich_monthly.set_index(['Company', 'year', 'month'])
print('Rich monthly panel:', len(rich_monthly), 'rows')

def build_monthly_wide_rich(predictor):
    s = rich_monthly_lookup[predictor].reset_index()
    w = s.pivot_table(index=['Company', 'year'], columns='month', values=predictor)
    w.columns = [f'm{mm}' for mm in range(1, 13)]
    return w.reset_index()

MONTHLY_WIDE_RICH = {p: build_monthly_wide_rich(p) for p in RICH_MIDAS_PREDICTORS}
MONTHLY_WIDE.update(MONTHLY_WIDE_RICH)

_cov = rich_monthly.groupby('year')[RICH_MIDAS_PREDICTORS].apply(lambda g: g.notna().mean().mean())
display(_cov.tail(6).round(3).to_frame('mean predictor coverage'))

Rich predictors (30): ['DE10YT_Yield', 'IT10YT_Yield', 'EURUSD_Close', 'EURUSD_Volume', 'FTMIB_Close', 'FTMIB_Volume', 'GVZ_Close', 'Brent_Close', 'Brent_Volume', 'OVX_Close', 'VIX_Close', 'Gold_Close', 'YC__level', 'YC__slope', 'YC__curv', 'TTF_Gas_Close', 'NFCI', 'Euribor_3M', 'IT_Inflation', 'IT_Unemployment', 'PCOPPUSDM', 'PALUMUSDM', 'IT_GDP', 'Close_logret', 'ff_HML', 'ff_Mom', 'Volume_log', 'excess_return', 'volatility', 'term_spread']
Rich monthly panel: 29565 rows


,mean predictor coverage
year,
2020,1.0
2021,1.0
2022,1.0
2023,1.0
2024,1.0
2025,1.0


### 3.2 Per-predictor U-MIDAS ridge forecasts + DMSFE/EQUAL combinations

In [ ]:
RICH_MODEL = 'MIDAS_RIDGE_FULL'

def run_model_set_rich(years, train_end_override=None, winsor_mode='train', predictors=None):
    predictors = RICH_MIDAS_PREDICTORS if predictors is None else predictors
    forecasts, selections = [], []
    for target_key, _, _, _ in TARGETS:
        for predictor in predictors:
            f, s = make_forecasts(target_key, predictor, years, train_end_override, winsor_mode, ridge=True)
            forecasts.append(f); selections.append(s)
    out = pd.concat(forecasts, ignore_index=True)
    out['model'] = RICH_MODEL          # distinguish from the paper-8 MIDAS_RIDGE
    return out, pd.concat(selections, ignore_index=True)

def add_combinations_general(forecasts, midas_model, predictors, history=None):
    """add_combinations parameterized by model label and predictor set (the
    original is kept untouched for reproducibility). Returns (forecasts, weights)."""
    forecasts = forecasts.copy()
    history = forecasts if history is None else history
    combined, weight_rows = [], []
    for target_key, _, _, _ in TARGETS:
        midas = forecasts[forecasts['target'].eq(target_key) & forecasts['model'].eq(midas_model)]
        hist_midas = history[history['target'].eq(target_key) & history['model'].eq(midas_model)]
        for year in sorted(midas['year'].unique()):
            history_years = [y for y in sorted(hist_midas['year'].unique()) if y < year][-DMSFE_LOOKBACK:]
            scores = {}
            for predictor in predictors:
                hist = hist_midas[hist_midas['predictor'].eq(predictor) & hist_midas['year'].isin(history_years)]
                if hist.empty:
                    continue
                by_year = (hist.assign(sq_error=(hist['actual'] - hist['prediction']) ** 2)
                           .dropna(subset=['sq_error']).groupby('year')['sq_error'].mean())
                score = sum(DMSFE_DELTA ** (len(history_years) - i) * by_year.get(y, np.nan)
                            for i, y in enumerate(history_years) if np.isfinite(by_year.get(y, np.nan)))
                if np.isfinite(score):
                    scores[predictor] = score
            pool = [p for p, v in scores.items() if v > 0]
            if not pool:
                continue
            if len(history_years) >= MIN_DMSFE_HISTORY:
                inverse = {p: 1.0 / scores[p] for p in pool}
                total = sum(inverse.values())
                weights = {p: w / total for p, w in inverse.items()}
                if max(weights.values()) > MAX_COMBO_WEIGHT:
                    weights = {p: min(w, MAX_COMBO_WEIGHT) for p, w in weights.items()}
                    wsum = sum(weights.values())
                    weights = {p: w / wsum for p, w in weights.items()}
            else:
                weights = {p: 1.0 for p in pool}
            for p, w in weights.items():
                weight_rows.append({'target': target_key, 'year': int(year), 'predictor': p, 'weight': w})
            current = midas[midas['year'].eq(year)]
            for company, group in current.groupby('Company'):
                group = group.set_index('predictor')
                valid = [p for p in weights if p in group.index and np.isfinite(group.loc[p, 'prediction'])]
                if not valid:
                    continue
                actual = float(group.loc[valid[0], 'actual'])
                wsum = sum(weights[p] for p in valid)
                dmsfe_pred = sum(weights[p] * group.loc[p, 'prediction'] for p in valid) / wsum
                equal_pred = float(group.loc[valid, 'prediction'].mean())
                combined.append({'target': target_key, 'model': f'{midas_model}_DMSFE', 'predictor': 'combination',
                                 'Company': company, 'year': int(year), 'actual': actual,
                                 'prediction': dmsfe_pred, 'n_predictors': len(valid)})
                combined.append({'target': target_key, 'model': f'{midas_model}_EQUAL', 'predictor': 'combination',
                                 'Company': company, 'year': int(year), 'actual': actual,
                                 'prediction': equal_pred, 'n_predictors': len(valid)})
    return pd.concat([forecasts, pd.DataFrame(combined)], ignore_index=True), pd.DataFrame(weight_rows)

print('Rolling rich forecasts (2012-2025) ...')
rich_rolling, rich_rolling_sel = run_model_set_rich(range(ROLLING_START_YEAR, int(annual_global.year.max()) + 1))
rich_rolling, rich_weights_rolling = add_combinations_general(rich_rolling, RICH_MODEL, RICH_MIDAS_PREDICTORS)

print('Fixed holdout rich forecasts (train <= 2022) ...')
rich_holdout, rich_holdout_sel = run_model_set_rich(HOLDOUT_YEARS, train_end_override=TRAIN_END)
rich_holdout, rich_weights_holdout = add_combinations_general(
    rich_holdout, RICH_MODEL, RICH_MIDAS_PREDICTORS, history=rich_rolling)

print('Rolling rich rows:', len(rich_rolling), '| holdout rich rows:', len(rich_holdout))

Rolling rich forecasts (2012-2025) ...


Fixed holdout rich forecasts (train <= 2022) ...


Rolling rich rows: 188410 | holdout rich rows: 32492


### 3.3 Pooled sensitivity: all predictors in one regression (exploratory)

In [ ]:
import warnings as _w
_w.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
def build_design_multi(rows, target_key, predictors, ar_lag, annual_lookup):
    y_col = f'd{target_key}'
    df = rows[['Company', 'year', y_col]].dropna(subset=[y_col]).copy()
    if df.empty:
        return np.empty((0, 0)), np.empty(0), []
    df = df.sort_values(['Company', 'year']).reset_index(drop=True)
    al = annual_lookup[[y_col]].reset_index().rename(columns={y_col: 'value'})
    cols = []
    for lag in range(1, ar_lag + 1):
        lag_key = al.assign(year=al['year'] + lag)
        name = f'lag{lag}'
        df[name] = df[['Company', 'year']].merge(lag_key, on=['Company', 'year'], how='left')['value']
        cols.append(name)
    new_cols = {}
    for predictor in predictors:
        w = MONTHLY_WIDE[predictor]
        m = df[['Company', 'year']].merge(w, on=['Company', 'year'], how='left')
        for mm in range(1, 13):
            name = f'{predictor}_m{mm}'
            new_cols[name] = m[f'm{mm}'].to_numpy()
            cols.append(name)
    if new_cols:
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    X = df[cols].to_numpy(float)
    valid = np.isfinite(X).all(axis=1) & np.isfinite(df[y_col].to_numpy(float))
    keys_out = [(c, int(y)) for c, y in zip(df['Company'].to_numpy()[valid], df['year'].to_numpy()[valid])]
    return X[valid], df[y_col].to_numpy(float)[valid], keys_out

ALL_MODEL = 'MIDAS_RIDGE_ALL'

def fit_ridge_multi(train_rows, target_key, annual_lookup, predictors=None, ar_lag=1):
    predictors = RICH_MIDAS_PREDICTORS if predictors is None else predictors
    X, y, _ = build_design_multi(train_rows, target_key, predictors, ar_lag, annual_lookup)
    if len(y) <= X.shape[1] + 2 or len(y) < MIN_TRAIN_ROWS:
        return None
    cv = RidgeCV(alphas=np.logspace(-3, 3, 15)).fit(X, y)
    clip_lo, clip_hi = np.nanpercentile(y, [1, 99])
    return {'intercept': float(cv.intercept_), 'coef': cv.coef_, 'ar_lag': ar_lag,
            'n_train': len(y), 'alpha': float(cv.alpha_), 'method': 'ridge',
            'clip_lo': float(clip_lo), 'clip_hi': float(clip_hi)}

def run_pooled(years, train_end_override=None, winsor_mode='train'):
    output = []
    for target_key, _, _, _ in TARGETS:
        for year in years:
            train_end = year - 1 if train_end_override is None else train_end_override
            frame = build_fold_frame(annual_global, train_end, winsor_mode)
            fold_annual_lookup = frame.set_index(['Company', 'year'])
            target_rows = frame[['Company', 'year', f'd{target_key}']].dropna().copy()
            train_rows = target_rows[target_rows['year'] <= train_end]
            test_rows = target_rows[target_rows['year'] == year]
            if len(train_rows) < MIN_TRAIN_ROWS or test_rows.empty:
                continue
            model = fit_ridge_multi(train_rows, target_key, fold_annual_lookup)
            if model is None:
                continue
            X, _, keys = build_design_multi(test_rows, target_key, RICH_MIDAS_PREDICTORS,
                                            model['ar_lag'], fold_annual_lookup)
            if len(X) == 0:
                continue
            preds = dict(zip(keys, model['intercept'] + X @ model['coef']))
            for _, row in test_rows.iterrows():
                key = (row['Company'], int(row['year']))
                if key not in preds:
                    continue
                p = float(preds[key])
                if CLIP_PREDICTIONS:
                    p = float(np.clip(p, model['clip_lo'], model['clip_hi']))
                output.append({'target': target_key, 'model': ALL_MODEL, 'predictor': 'pooled',
                               'Company': row['Company'], 'year': int(year),
                               'actual': float(row[f'd{target_key}']), 'prediction': p,
                               'winsor': winsor_mode, 'method': 'ridge'})
    return pd.DataFrame(output)

pooled_rolling = run_pooled(range(ROLLING_START_YEAR, int(annual_global.year.max()) + 1))
pooled_holdout = run_pooled(HOLDOUT_YEARS, train_end_override=TRAIN_END)
print('Pooled rolling rows:', len(pooled_rolling), '| holdout pooled rows:', len(pooled_holdout))
if len(pooled_rolling):
    print('Pooled fit coverage by year (all 3 targets):')
    print(pooled_rolling.groupby('year').size().to_string())

Pooled rolling rows: 4707 | holdout pooled rows: 1003
Pooled fit coverage by year (all 3 targets):
year
2015    341
2016    357
2017    375
2018    382
2019    416
2020    430
2021    449
2022    463
2023    491
2024    504
2025    499


### 3.4 Fixed-holdout forecasts (train ≤ 2022 for both 2024 and 2025)

In [11]:
holdout_forecasts, holdout_selections = run_model_set(HOLDOUT_YEARS, train_end_override=TRAIN_END, winsor_mode='train')
holdout_forecasts_r, holdout_selections_r = run_model_set(HOLDOUT_YEARS, train_end_override=TRAIN_END, winsor_mode='train', ridge=True)
holdout_forecasts = pd.concat([holdout_forecasts, holdout_forecasts_r], ignore_index=True)
holdout_forecasts = add_combinations(holdout_forecasts, history=rolling_forecasts)
holdout_forecasts = add_persistence(holdout_forecasts)


## 4. Headline results — fixed holdout 2024–2025

One table, Δ-ratio space, common AR-intersection subset (n = 308/306 company-years): mean absolute error, correlation, R², and the paper's MABER (median absolute error ratio vs AR; below 1 = better than AR). The pooled `MIDAS_RIDGE_ALL` is excluded (exploratory; see §5–§6).

In [12]:
def metric_table_gen(forecasts, years, models):
    """metric_table parameterized by model list (original kept untouched)."""
    f = forecasts[forecasts['year'].isin(years) & forecasts['model'].isin(models)]
    f = f.dropna(subset=['actual', 'prediction']).copy()
    rows = []
    for target, tf in f.groupby('target'):
        common = common_keys(f, target)
        for model, group in tf.groupby('model'):
            group = group[[(c, y) in common for c, y in zip(group['Company'], group['year'])]]
            if group.empty:
                continue
            error = group['prediction'].to_numpy() - group['actual'].to_numpy()
            actual = group['actual'].to_numpy()
            prediction = group['prediction'].to_numpy()
            corr = np.corrcoef(actual, prediction)[0, 1] if len(group) > 1 else np.nan
            r2 = 1.0 - np.mean(error ** 2) / (np.var(actual) + 1e-12)
            demeaned = group.copy()
            demeaned['actual'] -= demeaned.groupby('year')['actual'].transform('mean')
            demeaned['prediction'] -= demeaned.groupby('year')['prediction'].transform('mean')
            corr_yd = np.corrcoef(demeaned['actual'], demeaned['prediction'])[0, 1]
            r2_yd = 1.0 - np.mean((demeaned['actual'] - demeaned['prediction']) ** 2) / (np.var(demeaned['actual']) + 1e-12)
            rows.append({'target': target, 'model': model, 'n': len(group), 'MAE': np.mean(np.abs(error)),
                         'corr': corr, 'R2': r2, 'corr_year_demeaned': corr_yd, 'R2_year_demeaned': r2_yd})
    return pd.DataFrame(rows)

def maber_table_gen(forecasts, years, models, reference='AR'):
    """maber_table parameterized by model list (original kept untouched)."""
    f = forecasts[forecasts['year'].isin(years) & forecasts['model'].isin(models)].copy()
    rows = []
    for target in f['target'].unique():
        common = common_keys(f, target)
        ref = (f[f['model'].eq(reference) & f['target'].eq(target)][['Company', 'year', 'prediction']]
               .rename(columns={'prediction': 'ref_prediction'})
               .dropna(subset=['ref_prediction']).drop_duplicates(['Company', 'year']))
        for model, group in f[f['model'].ne(reference) & f['target'].eq(target)].groupby('model'):
            merged = (group.dropna(subset=['actual', 'prediction'])
                      .merge(ref, on=['Company', 'year'], how='inner'))
            merged = merged[[(c, y) in common for c, y in zip(merged['Company'], merged['year'])]]
            if merged.empty:
                continue
            numerator = np.median(np.abs(merged['actual'] - merged['prediction']))
            denominator = np.median(np.abs(merged['actual'] - merged['ref_prediction']))
            rows.append({'target': target, 'model': model,
                         f'MABER_vs_{reference}': numerator / denominator if denominator > 0 else np.nan,
                         'n': len(merged)})
    return pd.DataFrame(rows)

RICH_HEADLINE = ['Persistence (no-change)', 'AR', 'MIDAS_RIDGE_DMSFE',
                 'MIDAS_RIDGE_FULL_DMSFE', 'MIDAS_RIDGE_FULL_EQUAL']
holdout_all = pd.concat([holdout_forecasts, rich_holdout], ignore_index=True)

_headline = metric_table_gen(holdout_all, HOLDOUT_YEARS, RICH_HEADLINE)
_maber = maber_table_gen(holdout_all, HOLDOUT_YEARS, RICH_HEADLINE)
headline = _headline.merge(_maber[['target', 'model', 'MABER_vs_AR']], on=['target', 'model'], how='left')
headline.loc[headline['model'].eq('AR'), 'MABER_vs_AR'] = 1.0  # reference
print('=== Fixed holdout 2024-2025 (train <= 2022, train-only winsor; common AR-intersection subset) ===')
display(headline.sort_values(['target', 'MAE']).round(4))

=== Fixed holdout 2024-2025 (train <= 2022, train-only winsor; common AR-intersection subset) ===


,target,model,n,MAE,corr,R2,corr_year_demeaned,R2_year_demeaned,MABER_vs_AR
0,ebitda_ns,AR,308,0.0641,0.3947,0.1530,0.3918,0.1526,1.0000
2,ebitda_ns,MIDAS_RIDGE_FULL_DMSFE,308,0.0642,0.5011,0.2390,0.4988,0.2375,1.0384
4,ebitda_ns,Persistence (no-change),308,0.0656,NaN,-0.0004,NaN,0.0000,0.8888
1,ebitda_ns,MIDAS_RIDGE_DMSFE,308,0.0726,0.4629,0.2088,0.5085,0.2465,1.7370
3,ebitda_ns,MIDAS_RIDGE_FULL_EQUAL,308,0.0887,0.4836,0.0962,0.4934,0.2240,2.6348
5,ni_ns,AR,308,0.0605,0.3604,0.1243,0.3593,0.1239,1.0000
9,ni_ns,Persistence (no-change),308,0.0610,NaN,-0.0002,NaN,0.0000,0.8433
7,ni_ns,MIDAS_RIDGE_FULL_DMSFE,308,0.0641,0.4061,0.1515,0.4114,0.1595,1.2796
8,ni_ns,MIDAS_RIDGE_FULL_EQUAL,308,0.0675,0.4067,0.1462,0.4092,0.1644,1.7122
6,ni_ns,MIDAS_RIDGE_DMSFE,308,0.0709,0.3812,0.1237,0.4107,0.1606,1.7825


## 5. Comparison with the LSTM — identical test set

### 5.1 Exact test-set replication

In [ ]:
# --- Replicate the exact LSTM test set (297 firms, 2024-2025) ---
from collections import defaultdict as _defaultdict

_LT_TARGETS = ['EBITDA', 'Net_Income', 'ROA']
_LT_TEST_CUTOFF = 2023
_LT_WINSOR_Q = (0.005, 0.995)
_LT_DENOM = {'EBITDA': 'Net_Sales', 'Net_Income': 'Net_Sales', 'ROA': 'Total_Assets'}
_LT_NUM = {'EBITDA': 'EBITDA', 'Net_Income': 'Net_Income', 'ROA': 'Net_Income'}
_LT_PRIOR_COLS = ['Prior_EBITDA_SL', 'Prior_NI_SL', 'Prior_ROA', 'Leverage', 'Cash_Ratio',
                  'Size_SL', 'Prior_Net_Sales_SL', 'Prior_OpEx_SL', 'Prior_FCF_Per_Share_SL']

_lt = pd.read_csv(DATA_PATH)
_lt['Date'] = pd.to_datetime(_lt['Date'])
_lt = _lt.sort_values(['Company', 'Date']).reset_index(drop=True)
_lt['year'] = _lt['Date'].dt.year
_lt['has_targets'] = _lt[_LT_TARGETS].notna().any(axis=1)
_lt_records = _lt[_lt['has_targets']].copy()

_lt_exclude = ({'Date', 'Company', 'year', 'has_targets'} | set(_LT_TARGETS)
               | set(_LT_PRIOR_COLS)
               | {'Total_Assets', 'Total_Debt', 'Cash', 'Net_Sales', 'Operating_Expenses', 'FCF_Per_Share'}
               | {'MC_MIB', 'MC_MID', 'MC_SMALL', 'Sector'})
_lt_features = [c for c in _lt.columns
                if c not in _lt_exclude and pd.api.types.is_numeric_dtype(_lt[c])]

_lt_recs = []
for _t in _LT_TARGETS:
    _sub = _lt_records[_lt_records[_t].notna()][['Company', 'year', _LT_NUM[_t], _LT_DENOM[_t]]].copy()
    _den = _sub[_LT_DENOM[_t]].astype(float)
    _pos_med = _sub.assign(_den=_den)[lambda d: d['_den'] > 0].groupby('Company')['_den'].median()
    _med = _sub['Company'].map(_pos_med)
    _credible = (_den > 0) & (_den >= DENOM_REL_FLOOR * _med)
    _sub['ratio'] = np.where(_credible, _sub[_LT_NUM[_t]].astype(float) / _den, np.nan)
    _sub['target'] = _t
    _lt_recs.append(_sub[['Company', 'year', 'target', 'ratio']])
_lt_panel = pd.concat(_lt_recs, ignore_index=True)

for _t in _LT_TARGETS:
    _m = _lt_panel['target'] == _t
    _r_train = _lt_panel.loc[_m & (_lt_panel['year'] <= _LT_TEST_CUTOFF), 'ratio'].to_numpy(float)
    _lo, _hi = np.nanpercentile(_r_train, [_LT_WINSOR_Q[0] * 100, _LT_WINSOR_Q[1] * 100])
    _lt_panel.loc[_m, 'ratio'] = np.clip(_lt_panel.loc[_m, 'ratio'], _lo, _hi)

_lt_panel = _lt_panel.sort_values(['Company', 'target', 'year']).reset_index(drop=True)
_lt_pg = _lt_panel.groupby(['Company', 'target'])
_lt_panel['ratio_prev'] = _lt_pg['ratio'].shift(1)
_lt_panel['d_ratio'] = _lt_panel['ratio'] - _lt_panel['ratio_prev']
_lt_panel['d_ratio_prev'] = _lt_pg['d_ratio'].shift(1)

_lt_label, _lt_lag = _defaultdict(dict), _defaultdict(dict)
for _row in _lt_panel.itertuples(index=False):
    _key = (_row.Company, int(_row.year))
    if np.isfinite(_row.d_ratio):
        _lt_label[_key][_row.target] = float(_row.d_ratio)
    if np.isfinite(_row.ratio_prev) and np.isfinite(_row.d_ratio_prev):
        _lt_lag[_key][_row.target] = (float(_row.ratio_prev), float(_row.d_ratio_prev))

# ratio at year Y is ratio_prev for sample year Y+1
_lt_ratio_prev_map = {}
for _row in _lt_panel.itertuples(index=False):
    if np.isfinite(_row.ratio):
        _lt_ratio_prev_map[(_row.Company, _row.target, int(_row.year) + 1)] = float(_row.ratio)

_lt_static_cols = ['MC_MIB', 'MC_MID', 'MC_SMALL'] + _LT_PRIOR_COLS
_lt_keys = set()
for _company in _lt['Company'].unique():
    _c_rec = _lt_records[_lt_records['Company'] == _company]
    _c_data = _lt[_lt['Company'] == _company]
    for _, _yer in _c_rec.iterrows():
        _year = int(_yer['year'])
        _key = (_company, _year)
        if _key not in _lt_label or len(_lt_label[_key]) < len(_LT_TARGETS):
            continue
        if _key not in _lt_lag or len(_lt_lag[_key]) < len(_LT_TARGETS):
            continue
        _yed = _yer['Date']
        _window = _c_data[(_c_data['Date'] > _yed - pd.Timedelta(days=365))
                          & (_c_data['Date'] <= _yed)].copy()
        if len(_window) < 200:
            continue
        _window[_lt_features] = _window[_lt_features].ffill().bfill()
        if _window[_lt_features].isna().any().any():
            continue
        if _window[_lt_static_cols].iloc[-1].isna().any():
            continue
        _lt_keys.add(_key)

LSTM_TEST_KEYS = sorted(k for k in _lt_keys if k[1] > _LT_TEST_CUTOFF)
_lt_per_year = pd.Series([y for _, y in LSTM_TEST_KEYS]).value_counts().sort_index().to_dict()
print(f'LSTM test set replicated: {len(LSTM_TEST_KEYS)} firms {_lt_per_year}')
assert len(LSTM_TEST_KEYS) == 297 and _lt_per_year == {2024: 149, 2025: 148}, \
    'LSTM test-set replication mismatch — do not compare samples.'

# --- Meta aligned with LSTM_TEST_KEYS (same construction as build_test_meta) ---
_lt_ann = (pd.read_csv(DATA_PATH)
           .assign(Date=lambda d: pd.to_datetime(d['Date']), year=lambda d: d['Date'].dt.year))
_lt_ann = (_lt_ann[_lt_ann[_LT_TARGETS].notna().any(axis=1)]
           .sort_values(['Company', 'Date'])
           .drop_duplicates(['Company', 'year'], keep='last')
           .set_index(['Company', 'year']))

_N = len(LSTM_TEST_KEYS)
lt_ratio_prev = np.full((_N, 3), np.nan)
lt_net_sales = np.full(_N, np.nan)
lt_truth = np.full((_N, 3), np.nan)
for _i, (_c, _y) in enumerate(LSTM_TEST_KEYS):
    if (_c, _y) in _lt_ann.index:
        lt_net_sales[_i] = _lt_ann.loc[(_c, _y), 'Net_Sales']
        for _j, _t in enumerate(_LT_TARGETS):
            lt_truth[_i, _j] = _lt_ann.loc[(_c, _y), _t]
    for _j, _t in enumerate(_LT_TARGETS):
        lt_ratio_prev[_i, _j] = _lt_ratio_prev_map.get((_c, _t, _y), np.nan)

def lstm_style_raw_metrics(delta_by_key):
    """delta_by_key: {(Company, year): array(3) delta-ratio preds, NaN if missing}.
    Maps back to raw units and returns per-target rows (same logic as the LSTM
    notebook's raw_mae_delta_ratio, including its validity mask)."""
    _delta = np.vstack([delta_by_key.get(k, np.full(3, np.nan)) for k in LSTM_TEST_KEYS])
    _raw_pred = np.stack([
        (lt_ratio_prev[:, 0] + _delta[:, 0]) * lt_net_sales,
        (lt_ratio_prev[:, 1] + _delta[:, 1]) * lt_net_sales,
        lt_ratio_prev[:, 2] + _delta[:, 2],
    ], axis=1)
    _valid = (np.isfinite(_raw_pred) & np.isfinite(lt_truth)
              & (lt_net_sales[:, None] > 0) & np.isfinite(lt_ratio_prev))
    _rows = []
    for _j, _t in enumerate(_LT_TARGETS):
        _m = _valid[:, _j]
        if not _m.any():
            _rows.append({'target': _t, 'n': 0, 'MAE_delta': np.nan,
                          'nMAE_pct': np.nan, 'R2': np.nan})
            continue
        _p, _a = _raw_pred[_m, _j], lt_truth[_m, _j]
        _mae = float(np.mean(np.abs(_p - _a)))
        _ss_res = float(np.sum((_p - _a) ** 2))
        _ss_tot = float(np.sum((_a - _a.mean()) ** 2))
        _rows.append({'target': _t, 'n': int(_m.sum()),
                      'MAE_delta': float(np.mean(np.abs(_delta[_m, _j]))),
                      'nMAE_pct': 100.0 * _mae / float(np.mean(np.abs(_a))),
                      'R2': 1.0 - _ss_res / _ss_tot if _ss_tot > 0 else np.nan})
    return _rows

_lt_key_set = set(LSTM_TEST_KEYS)
# this notebook's target keys -> LSTM target order
_lt_t_idx = {'ebitda_ns': 0, 'ni_ns': 1, 'ni_ta': 2}

print('Replication helpers ready (LSTM test keys, raw meta, metric mapper).')

LSTM test set replicated: 297 firms {2024: 149, 2025: 148}


Replication helpers ready (LSTM test keys, raw meta, metric mapper).


### 5.2 Raw-scale nMAE%/R² — all models in one table

In [14]:
# --- nMAE%/R2 on the exact 297-firm LSTM test set: all model families ---
NMAE_MODELS = ['Persistence (no-change)', 'AR', 'MIDAS_RIDGE_DMSFE',
               'MIDAS_RIDGE_FULL_DMSFE', 'MIDAS_RIDGE_FULL_EQUAL', 'MIDAS_RIDGE_ALL']
holdout_all_full = pd.concat([holdout_forecasts, rich_holdout, pooled_holdout], ignore_index=True)
nmae_rows = []
for _mdl in NMAE_MODELS:
    if _mdl == 'Persistence (no-change)':
        _deltas = {k: np.zeros(3) for k in LSTM_TEST_KEYS}
    else:
        _src = holdout_all_full[holdout_all_full['model'].eq(_mdl) & holdout_all_full['year'].isin(HOLDOUT_YEARS)]
        _deltas = {}
        for _r in _src.itertuples(index=False):
            _key = (_r.Company, int(_r.year))
            if _key not in _lt_key_set or _r.target not in _lt_t_idx:
                continue
            _arr = _deltas.setdefault(_key, np.full(3, np.nan))
            _arr[_lt_t_idx[_r.target]] = _r.prediction
    for _row in lstm_style_raw_metrics(_deltas):
        nmae_rows.append({'model': _mdl, 'nMAE_std': np.nan, 'R2_std': np.nan, **_row})

_lstm_ref = pd.DataFrame([
    {'model': 'StackedLSTM L=4 (ref, 5 seeds)', 'target': _t, 'n': 297, 'MAE_delta': np.nan,
     'nMAE_pct': _nm, 'nMAE_std': _nms, 'R2': _r2, 'R2_std': _r2s}
    for _t, (_nm, _nms, _r2, _r2s) in {
        'EBITDA':     (32.0, 1.0, 0.384, 0.042),
        'Net_Income': (61.0, 1.1, -0.279, 0.021),
        'ROA':        (53.6, 0.5, 0.427, 0.008),
    }.items()
])
nmae_table = pd.concat([pd.DataFrame(nmae_rows), _lstm_ref], ignore_index=True)
nmae_table = nmae_table[['model', 'target', 'n', 'MAE_delta', 'nMAE_pct', 'nMAE_std', 'R2', 'R2_std']]
print(f'=== Raw-scale metrics on the exact LSTM test set ({len(LSTM_TEST_KEYS)} firms, 2024-2025, train <= 2022) ===')
display(nmae_table.round(4))
_persist_check = nmae_table[nmae_table['model'] == 'Persistence (no-change)'].set_index('target')
for _t, _v in {'EBITDA': 32.9, 'Net_Income': 66.0, 'ROA': 55.9}.items():
    assert abs(_persist_check.loc[_t, 'nMAE_pct'] - _v) < 0.05, (_t, _persist_check.loc[_t, 'nMAE_pct'])
print('Persistence cross-check vs LSTM notebook: OK (32.9% / 66.0% / 55.9%).')

=== Raw-scale metrics on the exact LSTM test set (297 firms, 2024-2025, train <= 2022) ===


,model,target,n,MAE_delta,nMAE_pct,nMAE_std,R2,R2_std
0,Persistence (no-change),EBITDA,297,0.0000,32.8818,NaN,0.3849,NaN
1,Persistence (no-change),Net_Income,297,0.0000,66.0203,NaN,-0.3908,NaN
2,Persistence (no-change),ROA,297,0.0000,55.9132,NaN,0.3906,NaN
3,AR,EBITDA,297,0.0223,35.2070,NaN,0.2073,NaN
4,AR,Net_Income,297,0.0268,74.2603,NaN,-0.8681,NaN
5,AR,ROA,297,0.0093,54.3341,NaN,0.4038,NaN
6,MIDAS_RIDGE_DMSFE,EBITDA,273,0.0378,43.9685,NaN,-0.0376,NaN
7,MIDAS_RIDGE_DMSFE,Net_Income,273,0.0453,86.1902,NaN,-1.3063,NaN
8,MIDAS_RIDGE_DMSFE,ROA,273,0.0172,54.4513,NaN,0.4712,NaN
9,MIDAS_RIDGE_FULL_DMSFE,EBITDA,297,0.0313,39.1706,NaN,0.1358,NaN


Persistence cross-check vs LSTM notebook: OK (32.9% / 66.0% / 55.9%).


## 6. Robustness — computed and stored, not displayed

All secondary views are computed once and stored in variables for inspection.

In [15]:
# --- Robustness: compute and store (no displays) ---
ROLLING_YEARS = range(ROLLING_START_YEAR, int(annual_global.year.max()) + 1)
rolling_all = pd.concat([rolling_forecasts, rich_rolling, pooled_rolling], ignore_index=True)
tbl_rolling = metric_table_gen(rolling_all, ROLLING_YEARS, RICH_HEADLINE)

tbl_per_year = {yr: metric_table_gen(holdout_all, [yr], RICH_HEADLINE) for yr in HOLDOUT_YEARS}

_rich_pred_frame = pd.concat([
    holdout_forecasts[holdout_forecasts['model'].eq('AR')],
    rich_holdout[rich_holdout['model'].eq(RICH_MODEL)].assign(model='MIDAS_RIDGE'),
], ignore_index=True)
tbl_pred = metric_table_predictor(_rich_pred_frame, HOLDOUT_YEARS).sort_values(['target', 'MAE'])

w_pred = (rich_weights_holdout[rich_weights_holdout['year'].isin(HOLDOUT_YEARS)]
          .groupby('predictor')['weight'].mean().sort_values(ascending=False))

sens_rich, _ = run_model_set_rich(HOLDOUT_YEARS, train_end_override=TRAIN_END, winsor_mode='global')
sens_rich, _ = add_combinations_general(sens_rich, RICH_MODEL, RICH_MIDAS_PREDICTORS, history=rich_rolling)
sens_all = pd.concat([holdout_forecasts, sens_rich], ignore_index=True)
tbl_winsor_rich = metric_table_gen(sens_all, HOLDOUT_YEARS, RICH_HEADLINE).sort_values(['target', 'MAE'])

sens_paper, _ = run_model_set(HOLDOUT_YEARS, train_end_override=TRAIN_END, winsor_mode='global')
sens_paper_r, _ = run_model_set(HOLDOUT_YEARS, train_end_override=TRAIN_END, winsor_mode='global', ridge=True)
sens_paper = add_persistence(add_combinations(
    pd.concat([sens_paper, sens_paper_r], ignore_index=True), history=rolling_forecasts))
tbl_winsor_paper = metric_table_gen(sens_paper, HOLDOUT_YEARS, HEADLINE_MODELS).sort_values(['target', 'MAE'])

tbl_pooled = metric_table_gen(holdout_all_full, HOLDOUT_YEARS, [ALL_MODEL])

print('Robustness results stored (not displayed):')
print('  tbl_rolling       rolling 2012-2025, headline models (common subset)')
print('  tbl_per_year      fixed-holdout MAE by year (2024 / 2025)')
print('  tbl_pred          per-predictor breakdown, rich set vs AR (holdout)')
print('  w_pred            mean DMSFE weight per predictor (holdout years)')
print('  tbl_winsor_rich   global-winsor sensitivity, rich combos')
print('  tbl_winsor_paper  global-winsor sensitivity, paper-8 combos')
print('  tbl_pooled        MIDAS_RIDGE_ALL on its own sample (exploratory)')

Robustness results stored (not displayed):
  tbl_rolling       rolling 2012-2025, headline models (common subset)
  tbl_per_year      fixed-holdout MAE by year (2024 / 2025)
  tbl_pred          per-predictor breakdown, rich set vs AR (holdout)
  w_pred            mean DMSFE weight per predictor (holdout years)
  tbl_winsor_rich   global-winsor sensitivity, rich combos
  tbl_winsor_paper  global-winsor sensitivity, paper-8 combos
  tbl_pooled        MIDAS_RIDGE_ALL on its own sample (exploratory)


**Robustness verdict**

- Over the full rolling period 2012–2025 the rich combination still trails persistence and AR (`tbl_rolling`: FULL_DMSFE 0.0942 / 0.1197 / 0.0402 vs persistence 0.0842 / 0.0860 / 0.0356) — its gains are concentrated in the recent holdout, so the §4 result should be read as period-specific.
- Global (full-sample) winsorization leaves AR and the rich combinations essentially unchanged (`tbl_winsor_*`); only the paper-8 equal-weight combination was ever fragile to the tail treatment.
- The DMSFE weights are economically interpretable (`w_pred`): return-type features earn the most (excess_return, volatility, EURUSD_Close, ff_HML, Close_logret ≈ 0.054–0.056), while price levels are starved (FTMIB_Volume 0.002, FTMIB_Close 0.004, Gold 0.005, Brent 0.007).
- Per predictor (`tbl_pred`): the useful new features are return- and labor-market variables (EURUSD_Close, IT_Unemployment, MABER ≈ 1.21); index levels remain toxic (MABER up to ~18× AR).

## 7. Diagnostics

In [16]:
print('=== Fit coverage per (predictor, model) over rolling runs (expected 42 per model) ===')
counts = rolling_selections.groupby(['model', 'predictor']).size().reset_index(name='count')
display(counts.pivot(index='predictor', columns='model', values='count').fillna(0).astype(int))

print('=== Monthly coverage by year (company-months vs expected) ===')
expected = monthly.groupby('year')['Company'].nunique() * 12
cov = monthly.groupby('year').size()
display(pd.DataFrame({'company_months': cov, 'expected': expected})
        .assign(pct=lambda d: (d['company_months'] / d['expected']).round(3)).tail(8))

print('=== Condition number of the 12-month block (2022 rows) ===')
sample_rows = annual_global[annual_global['year'] == 2022]
for predictor in MIDAS_PREDICTORS:
    X, y, _ = build_design(sample_rows, 'ebitda_ns', predictor, 1, 0, annual_lookup, monthly_lookup)
    if X.shape[0] > X.shape[1] + 1:
        print(f'{predictor:16s} n={X.shape[0]:4d} cond = {np.linalg.cond(X[:, 1:]):12.2f}')

=== Fit coverage per (predictor, model) over rolling runs (expected 42 per model) ===


model,AR,MIDAS,MIDAS_RIDGE
predictor,,,
Brent_Close,0,39,42
Euribor_3M,0,36,42
IT_Inflation,0,12,42
NFCI,0,36,42
YC__curv,0,39,42
excess_return,0,42,42
lagged_target,42,0,0
term_spread,0,39,42
volatility,0,42,42


=== Monthly coverage by year (company-months vs expected) ===


,company_months,expected,pct
year,,,
2018,1879,1908,0.985
2019,1947,1980,0.983
2020,2004,2016,0.994
2021,2046,2088,0.980
2022,2138,2160,0.990
2023,2203,2220,0.992
2024,2226,2232,0.997
2025,2232,2232,1.000


=== Condition number of the 12-month block (2022 rows) ===
excess_return    n= 154 cond =         2.73
volatility       n= 154 cond =        17.80
IT_Inflation     n= 154 cond = 3597135118502338277356329359295991635416552394032824275817145487766066257481280512964157508796667958595903150760184264952603565401652208434982714945077301920142439082885120.00
Brent_Close      n= 154 cond = 2375656289492924891136.00
Euribor_3M       n= 154 cond = 164421400853651398919938209434566656.00
term_spread      n= 154 cond = 9937787544251012218880.00
NFCI             n= 154 cond = 427247658197709616054272.00
YC__curv         n= 154 cond = 919749235877619564544.00
